# Preprocessing PTA Universitas Trunojoyo Madura

In [1]:
# Colab cell 1: mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install Sastrawi pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 51.0 MB/s eta 0:00:00


In [3]:
# Colab cell 2: impor libraries
import os
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from spellchecker import SpellChecker

# Pastikan download resource NLTK
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
# Colab cell 3: setup stopwords, stemmer, spellchecker custom
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from spellchecker import SpellChecker

# === Stopwords bahasa Indonesia ===
stop_words = set(stopwords.words("indonesian"))

# === Stemmer (Sastrawi) ===
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# === SpellChecker (tanpa bawaan bahasa, kita isi manual) ===
spell = SpellChecker(language=None)  # kosong, biar kita isi sendiri

# === Daftar kata baku khas abstrak skripsi ===
custom_words = [
    # Umum skripsi
    "penelitian","skripsi","tesis","disertasi",
    "tujuan","manfaat","kontribusi","implikasi",
    "metode","metodologi","pendekatan","teknik","algoritma",
    "analisis","klasifikasi","prediksi","peramalan","perancangan",
    "data","dataset","sampel","variabel","responden","instrumen",
    "observasi","kuisioner","wawancara","uji","validitas","reliabilitas",
    "hasil","temuan","pembahasan","diskusi","kesimpulan","saran",
    "akurasi","precision","recall","evaluasi","performa","f1",
    "sistem","aplikasi","implementasi","pengujian","simulasi",
    "studi","kasus","literatur","kajian","teori","model","framework"
]

# Masukkan kata ke spellchecker biar dianggap baku
spell.word_frequency.load_words(custom_words)

# === Normalisasi manual (singkatan → baku) ===
normalize_dict = {
    "gak": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "dr": "dari",
    "yg": "yang",
    "tp": "tetapi",
    "utk": "untuk",
    "biarpun": "meskipun",
    "acc": "akurasi",
    "resp": "responden",
    "var": "variabel",
    "metod": "metode"
}

def normalize_word(word):
    # cek normalisasi manual
    if word in normalize_dict:
        return normalize_dict[word]
    # cek spellchecker
    correction = spell.correction(word)
    return correction if correction is not None else word


In [5]:
# Colab cell 4: definisi fungsi preprocessing bertahap
def preprocessing_steps(text):
    original = text

    # Lowercase
    text_lower = original.lower()

    # Remove angka & simbol/tanda baca
    no_symbol = re.sub(r"\d+", " ", text_lower)
    no_symbol = no_symbol.translate(str.maketrans("", "", string.punctuation))

    # Tokenisasi
    tokens = nltk.word_tokenize(no_symbol)

    # Stopword removal
    no_stop = [w for w in tokens if w not in stop_words]

    # Spell correction / pembakuan ejaan
    corrected = [spell.correction(w) if spell.correction(w) is not None else w for w in no_stop]

    # Stemming
    stemmed = [stemmer.stem(w) for w in corrected]

    return {
        "before": original,
        "after_lower_no_symbol": " ".join(tokens),
        "after_stopword": " ".join(no_stop),
        "after_corrected": " ".join(corrected),
        "after_stemmed": " ".join(stemmed),
        "tokens_final": stemmed  # atau bisa pakai corrected / stemmed sesuai kebutuhan
    }

In [7]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
# Colab cell 5: load data & aplikasikan
# Ganti path ke folder-drive kamu
file_path = "/content/drive/MyDrive/Semester 7/pta.csv"
df = pd.read_csv(file_path, dtype=str)  # dtype=str supaya semua teks dianggap string

# Terapkan preprocessing ke kolom 'isi'
results = df["abstrak"].fillna("").apply(preprocessing_steps)

# Tambahkan kolom hasil ke dataframe
df["before"] = results.apply(lambda x: x["before"])
df["after_lower_no_symbol"] = results.apply(lambda x: x["after_lower_no_symbol"])
df["after_stopword"] = results.apply(lambda x: x["after_stopword"])
df["after_corrected"] = results.apply(lambda x: x["after_corrected"])
df["after_stemmed"] = results.apply(lambda x: x["after_stemmed"])
df["tokens_final"] = results.apply(lambda x: x["tokens_final"])


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Semester 7/pta_preprocessed.csv'

In [13]:
# Colab cell 6: simpan hasil ke Drive
output_path = "/content/drive/MyDrive/Semester 7/pta_prepocessed.csv"
df.to_csv(output_path, index=False)
print("Hasil preprocessing tersimpan di:", output_path)

Hasil preprocessing tersimpan di: /content/drive/MyDrive/Semester 7/pta_prepocessed.csv


In [14]:
# Tampilkan 5 berita teratas lengkap semua tahapan + tokenisasi
pd.set_option("display.max_colwidth", 200)
display(df[[
    "before",
    "after_lower_no_symbol",
    "after_stopword",
    "after_corrected",
    "after_stemmed",
    "tokens_final"
]].head(5))


,before,after_lower_no_symbol,after_stopword,after_corrected,after_stemmed,tokens_final
0,ABSTRAK\r\n\r\n Implementasi Fungsi Legislasi DPRD Kabupaten Bangkalan Periode 2009-2014 Dalam Pembentukan Peraturan Daerah menurut ketentuan Undang-Undang Dasar Negara Republik Indonesia Ta...,abstrak implementasi fungsi legislasi dprd kabupaten bangkalan periode dalam pembentukan peraturan daerah menurut ketentuan undangundang dasar negara republik indonesia tahun maupun undangundang r...,abstrak implementasi fungsi legislasi dprd kabupaten bangkalan periode pembentukan peraturan daerah ketentuan undangundang dasar negara republik indonesia undangundang republik indonesia nomor pem...,abstrak implementasi fungsi legislasi dprd kabupaten bangkalan periode pembentukan peraturan daerah ketentuan undangundang data negara republik indonesia undangundang republik indonesia nomor pemb...,abstrak implementasi fungsi legislasi dprd kabupaten bangkal periode bentuk atur daerah tentu undangundang data negara republik indonesia undangundang republik indonesia nomor bentuk atur perundan...,"[abstrak, implementasi, fungsi, legislasi, dprd, kabupaten, bangkal, periode, bentuk, atur, daerah, tentu, undangundang, data, negara, republik, indonesia, undangundang, republik, indonesia, nomor..."
1,"Badan Usaha Milik Negara (BUMN) adalah Badan usaha yang sebagian atau seluruh modalnya dimiliki oleh Negara yang berasal dari kekayaan negara yang dipisahkan. Namun, segala ketentuan yang berlaku ...",badan usaha milik negara bumn adalah badan usaha yang sebagian atau seluruh modalnya dimiliki oleh negara yang berasal dari kekayaan negara yang dipisahkan namun segala ketentuan yang berlaku pada...,badan usaha milik negara bumn badan usaha modalnya dimiliki negara berasal kekayaan negara dipisahkan ketentuan berlaku bumn persero mengacu undangundang perseroan terbatas kenyataannya direksi bu...,saran usaha milik negara bumn saran usaha modalnya dimiliki negara berasal kekayaan negara dipisahkan ketentuan berlaku bumn persero mengacu undangundang perseroan terbatas kenyataannya direksi bu...,saran usaha milik negara bumn saran usaha modal milik negara asal kaya negara pisah tentu laku bumn persero acu undangundang persero batas nyata direksi bumn persero mana direksi bertanggungjawab ...,"[saran, usaha, milik, negara, bumn, saran, usaha, modal, milik, negara, asal, kaya, negara, pisah, tentu, laku, bumn, persero, acu, undangundang, persero, batas, nyata, direksi, bumn, persero, man..."
2,"Kasus narkoba tidak henti-hentinya terdengar di media televisi, radio dan media cetak yang memberitakan tentang penyalahgunaan narkoba mulai dari penggunaan hingga peredaran gelapnya. Narkoba sang...",kasus narkoba tidak hentihentinya terdengar di media televisi radio dan media cetak yang memberitakan tentang penyalahgunaan narkoba mulai dari penggunaan hingga peredaran gelapnya narkoba sangat ...,narkoba hentihentinya terdengar media televisi radio media cetak memberitakan penyalahgunaan narkoba penggunaan peredaran gelapnya narkoba memiliki harga nominal orang menjadikan narkoba ladang bi...,narkoba hentihentinya terdengar media televisi radio media cetak memberitakan penyalahgunaan narkoba penggunaan peredaran gelapnya narkoba memiliki harga nominal orang menjadikan narkoba ladang bi...,narkoba hentihentinya dengar media televisi radio media cetak berita penyalahgunaan narkoba guna edar gelap narkoba milik harga nominal orang jadi narkoba ladang bisnis hirau dampak bahaya resiko ...,"[narkoba, hentihentinya, dengar, media, televisi, radio, media, cetak, berita, penyalahgunaan, narkoba, guna, edar, gelap, narkoba, milik, harga, nominal, orang, jadi, narkoba, ladang, bisnis, hir..."
3,Produk elektronik adalah suatu benda bergerak yang dihasilkan melalui proses produksi oleh pengusaha elektronik berupa Peralatan atau alat yang bekerja berdasarkan kerja elektronika. Produk elektr...,produk elektronik adalah suatu benda bergerak yang dihasilkan melalui proses produksi oleh pengusaha